# 总光时延迟 `T_GR`（加入 HM 列）

本 notebook 在原先仅使用 `T_PM` 的基础上，额外读取右侧表格中的 **HM 列**（按 `gps_time` 对齐），并计算：

- `T_GR_s = T_PM_s + T_HM_s`
- `rho_total_m = c0 * T_GR_s`

同时给出距离域自检：

- `rho_sum_m = rho_PM_m + rho_HM_m`
- `rho_diff_m = rho_total_m - rho_sum_m`

其中：
- `T_PM` 数据来自 `TPM_XLSX`
- `T_HM` 数据来自 `HM_XLSX`
- 两表通过 `gps_time` 做内连接对齐


In [17]:
import pandas as pd

C0 = 299792458.0

# ===== 输入文件 =====
# T_PM 结果表（原程序输出或已有结果）
TPM_XLSX = "Shapiro_TPM_GNI1B_D_emit_C_recv_d0.xlsx"

# HM 表（右边表格）
HM_XLSX = "TpMr_THM.xlsx"


In [18]:
# ===== 读取数据 =====
tpm = pd.read_excel(TPM_XLSX)
hm  = pd.read_excel(HM_XLSX)

print("TPM rows:", len(tpm), "cols:", list(tpm.columns))
print("HM  rows:", len(hm),  "cols:", list(hm.columns))

# ===== 兼容不同列名 =====
# HM 表中常见列名可能为：T_HM / HM / T_HM_s
hm_time_col = "gps_time"
if hm_time_col not in hm.columns:
    raise KeyError(f"HM 表中未找到时间列: {hm_time_col}")

hm_col_candidates = ["T_HM_s", "T_HM", "HM"]
hm_col = next((c for c in hm_col_candidates if c in hm.columns), None)
if hm_col is None:
    raise KeyError(
        "HM 表中未找到 HM 列。可接受列名之一："
        + ", ".join(hm_col_candidates)
    )

# 统一 HM 秒域列名
hm_use = hm[[hm_time_col, hm_col]].copy().rename(columns={hm_col: "T_HM_s"})

# 若 TPM 中没有 rho_PM_m，但有 T_PM_s，则补算
if "rho_PM_m" not in tpm.columns and "T_PM_s" in tpm.columns:
    tpm["rho_PM_m"] = C0 * tpm["T_PM_s"]

# 若 HM 中没有 rho_HM_m，则由 T_HM_s 补算
hm_use["rho_HM_m"] = C0 * hm_use["T_HM_s"]

# ===== 按 gps_time 对齐合并 =====
merged = pd.merge(tpm, hm_use, on="gps_time", how="inner")

print("Merged rows:", len(merged))

TPM rows: 86400 cols: ['gps_time', 'T_PM_s', 'rho_PM_m', 'dt_inst_s', 'dt_corr_s', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vC_mps', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2', 'r_e_x_m', 'r_e_y_m', 'r_e_z_m']
HM  rows: 86400 cols: ['gps_time', 'T_HM_s', 'rho_HM_m', 'dt_inst_s', 'dt_emit_s', 'dt_recv_corr_s', 'te_seconds', 'delta_t_sr_s', 're_x', 're_y', 're_z', 'rr_x', 'rr_y', 'rr_z', 'd0_x', 'd0_y', 'd0_z', 'd0_dot_vC_mps', 'd0_dot_vD_mps', 'aC_x_mps2', 'aC_y_mps2', 'aC_z_mps2', 'aC_mag_mps2', 'aD_x_mps2', 'aD_y_mps2', 'aD_z_mps2', 'aD_mag_mps2']
Merged rows: 86400


In [19]:
# ===== 计算 T_GR =====
if "T_PM_s" not in merged.columns:
    raise KeyError("TPM 表中未找到 T_PM_s 列，无法计算 T_GR。")

merged["T_GR_s"] = merged["T_PM_s"] - merged["T_HM_s"]
merged["rho_total_m"] = C0 * merged["T_GR_s"]

# 距离域自检
merged["rho_sum_m"] = merged["rho_PM_m"] + merged["rho_HM_m"]
merged["rho_diff_m"] = merged["rho_total_m"] - merged["rho_sum_m"]

# 如需保留旧名字，也同步给出
merged["delta_t_s"] = merged["T_GR_s"]

merged[[
    "gps_time",
    "T_PM_s", "T_HM_s", "T_GR_s",
    "rho_PM_m", "rho_HM_m", "rho_total_m",
    "rho_sum_m", "rho_diff_m"
]].head(10)

,gps_time,T_PM_s,T_HM_s,T_GR_s,rho_PM_m,rho_HM_m,rho_total_m,rho_sum_m,rho_diff_m
0,707659200,8.419040e-13,-1.831762e-16,8.420871e-13,0.000252,-5.491485e-08,0.000252,0.000252,1.098297e-07
1,707659201,8.419052e-13,-1.846429e-16,8.420898e-13,0.000252,-5.535454e-08,0.000252,0.000252,1.107091e-07
2,707659202,8.419064e-13,-1.861097e-16,8.420925e-13,0.000252,-5.579428e-08,0.000252,0.000252,1.115886e-07
3,707659203,8.419076e-13,-1.875767e-16,8.420952e-13,0.000252,-5.623408e-08,0.000252,0.000252,1.124682e-07
4,707659204,8.419088e-13,-1.890438e-16,8.420979e-13,0.000252,-5.667392e-08,0.000252,0.000252,1.133478e-07
5,707659205,8.419100e-13,-1.905112e-16,8.421005e-13,0.000252,-5.711381e-08,0.000252,0.000252,1.142276e-07
6,707659206,8.419112e-13,-1.919786e-16,8.421032e-13,0.000252,-5.755374e-08,0.000252,0.000252,1.151075e-07
7,707659207,8.419124e-13,-1.934462e-16,8.421058e-13,0.000252,-5.799371e-08,0.000252,0.000252,1.159874e-07
8,707659208,8.419136e-13,-1.949139e-16,8.421085e-13,0.000252,-5.843372e-08,0.000252,0.000252,1.168674e-07
9,707659209,8.419148e-13,-1.963817e-16,8.421111e-13,0.000252,-5.887376e-08,0.000252,0.000252,1.177475e-07


In [20]:
# ===== 输出结果 =====
OUT_XLSX = "LightTime_T_GR_TpMr.xlsx"
merged.to_excel(OUT_XLSX, index=False)

print("Wrote:", OUT_XLSX)
print("rows:", len(merged))
print("rho_diff_m abs max:", merged["rho_diff_m"].abs().max())

Wrote: LightTime_T_GR_TpMr.xlsx
rows: 86400
rho_diff_m abs max: 5.248309244056809e-07
